# Qari-OCR as a service, on a Colab GPU

Runs [Qari-OCR](https://huggingface.co/NAMAA-Space/Qari-OCR-v0.3-VL-2B-Instruct)
— Qwen2-VL fine-tuned on Arabic documents — behind an HTTP API, and exposes it
through an ngrok tunnel so a machine elsewhere can POST a PDF and get Arabic text back.

**Why bother.** Measured against this project's own corpus, Qari reads Arabic at
**0.063 word error rate** versus 0.172 for `tesseract-best`, and it keeps diacritics
and inline English that everything else mangles. It needs ~5 GB of VRAM, which a
2-vCPU server does not have and a free Colab T4 does.

**Run the cells in order.** Two need editing first: the ngrok token and the shared
secret in cell 4.

---

### Before you start

1. **Set the runtime to a GPU.** *Runtime → Change runtime type → T4 GPU*. On CPU
   this model is unusably slow and the notebook will refuse to continue.
2. **Get an ngrok authtoken** — free, from
   <https://dashboard.ngrok.com/get-started/your-authtoken>. Without one the tunnel
   closes after a couple of minutes.

### Know the limits before you depend on it

- A Colab session dies after ~12 hours, and sooner if the tab is idle. The tunnel URL
  dies with it.
- A free ngrok URL **changes every restart**. Whatever calls this has to be told the
  new one — see the last cell for how to fetch it.
- This sends your PDFs to Google's machines and through ngrok's. If that is not
  acceptable for a document, run `tesseract-best` locally instead.


## 1. Confirm there is a GPU


In [ ]:
import subprocess, sys

try:
    print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                          '--format=csv,noheader'],
                         capture_output=True, text=True, check=True).stdout.strip())
except Exception:
    sys.exit('No GPU. Runtime -> Change runtime type -> T4 GPU, then run this again.')


## 2. Install

Two or three minutes. `transformers` and `torch` are already on Colab; the rest are not.


In [ ]:
%pip install -q --upgrade "transformers>=4.49" accelerate qwen-vl-utils \
    pymupdf pillow fastapi "uvicorn[standard]" python-multipart pyngrok

# Colab preloads its own transformers. If pip replaced it, the version already
# imported into this kernel is the old one, and the mismatch surfaces later as
# a confusing error inside the model load rather than here.
import importlib.metadata as md

installed = md.version('transformers')
print(f'transformers {installed}')

import sys

if 'transformers' in sys.modules:
    print()
    print('  !! transformers was already imported in this kernel.')
    print('  !! Runtime -> Restart session, then run from cell 1 again.')
    print('  !! (Skipping this is the most common way this notebook fails.)')
else:
    print('ok -- nothing stale imported, carry on')


## 3. Settings

**Edit both values below.** The shared secret is not decoration: an ngrok URL is on
the public internet, and without it anyone who finds the address can spend your GPU
and read whatever they upload.


In [ ]:
# Paste your token from https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTHTOKEN = "PASTE_YOUR_NGROK_TOKEN_HERE"

# Invent a long random string. The client must send it as:  Authorization: Bearer <this>
API_SECRET = "PASTE_A_LONG_RANDOM_STRING_HERE"

PORT = 8888
MODEL_ID = "NAMAA-Space/Qari-OCR-v0.3-VL-2B-Instruct"

# 300 dpi is the floor OCR engines are trained around: below it the dots that
# separate  ب ت ث  start sharing a pixel.
RENDER_DPI = 300

# Qwen2-VL turns every 28x28 patch into a token. Too low and a dense page comes back
# as a few characters -- that is not a hypothetical, it happened at 1664 during
# benchmarking. Too high and prefill takes minutes. 6400 was measured to work.
MAX_PIXELS = 6400 * 28 * 28

assert not NGROK_AUTHTOKEN.startswith('PASTE'), 'set NGROK_AUTHTOKEN above'
assert not API_SECRET.startswith('PASTE'), 'set API_SECRET above'
assert len(API_SECRET) >= 24, 'use a longer secret; this one is guessable'
print('settings ok')


## 4. Load the model

~5 GB downloaded on the first run and cached for the session. Loading it here rather
than on the first request means the first PDF is not charged for the model load.


In [ ]:
import torch
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration

model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto'
)
model.eval()

processor = AutoProcessor.from_pretrained(
    MODEL_ID, min_pixels=256 * 28 * 28, max_pixels=MAX_PIXELS
)

print(f'{MODEL_ID} loaded on {model.device}')
print(f'VRAM allocated: {torch.cuda.memory_allocated() / 1e9:.1f} GB')


## 5. The extraction itself

Three details here are the difference between this matching the benchmark and
quietly diverging from it:

- **Greedy decoding** (`do_sample=False`). This is transcription, not writing.
  Sampling invents plausible Arabic, which is the one failure mode no downstream
  check would catch.
- **Markup stripped.** Qari v0.3 is trained to emit `<h1>`, `<u>` and friends. That is
  a real feature and it is not text anyone indexes — left in, it scored 0.233 WER
  while its Arabic was letter-perfect.
- **NFKC + bidi stripping.** PDF Arabic arrives as presentation-form glyphs
  (U+FB50-FDFF, U+FE70-FEFF) that render identically to typed letters and compare as
  different characters, so a query never matches until they are folded.


In [ ]:
import io, re, time, unicodedata
import pymupdf  # `import fitz` still works but is deprecated
from PIL import Image

PROMPT = (
    'Transcribe all the Arabic text in this document image exactly as written, '
    'preserving the original word spacing and line breaks. '
    'Output only the transcription, with no commentary, no translation and no '
    'markdown fences.'
)

BIDI = dict.fromkeys(map(ord, '\u200e\u200f\u202a\u202b\u202c\u202d\u202e'), None)


def strip_markup(text: str) -> str:
    """Drop the HTML the model is trained to emit; keep its line structure."""
    text = re.sub(r'<br\s*/?>', '\n', text, flags=re.IGNORECASE)
    text = re.sub(r'<[^>]+>', ' ', text)
    return re.sub(r'[ \t]{2,}', ' ', text).strip()


def normalize(text: str) -> str:
    """Fold to the codepoints a query is actually written in."""
    folded = unicodedata.normalize('NFKC', text).translate(BIDI)
    return re.sub(r'[ \t\u00a0]{2,}', ' ', folded)


def read_page(image: Image.Image, max_new_tokens: int = 2048) -> str:
    messages = [{'role': 'user', 'content': [
        {'type': 'image', 'image': image},
        {'type': 'text', 'text': PROMPT},
    ]}]
    prompt = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(text=[prompt], images=[image], return_tensors='pt').to(model.device)

    with torch.inference_mode():
        generated = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False
        )

    decoded = processor.decode(
        generated[0][inputs.input_ids.shape[1]:], skip_special_tokens=True
    )
    return normalize(strip_markup(decoded))


def read_pdf(data: bytes, first: int = 0, last: int | None = None) -> dict:
    document = pymupdf.open(stream=data, filetype='pdf')
    try:
        end = document.page_count if last is None else min(last + 1, document.page_count)
        pages, started = [], time.perf_counter()

        for number in range(first, end):
            pixmap = document[number].get_pixmap(dpi=RENDER_DPI)
            image = Image.open(io.BytesIO(pixmap.tobytes('png'))).convert('RGB')
            page_started = time.perf_counter()
            pages.append({
                'page': number,
                'text': read_page(image),
                'seconds': round(time.perf_counter() - page_started, 2),
            })

        return {
            'pages': pages,
            'page_count': document.page_count,
            'seconds': round(time.perf_counter() - started, 2),
        }
    finally:
        document.close()

print('extraction ready')


## 6. The API

`POST /ocr` takes a PDF and returns one entry per page.

A whole book in one request will exceed any sensible timeout, so `first_page` and
`last_page` exist and there is a hard cap on pages per request. Call it in batches.


In [ ]:
from fastapi import FastAPI, File, Form, Header, HTTPException, UploadFile

MAX_PAGES_PER_REQUEST = 20
MAX_UPLOAD_MB = 50

app = FastAPI(title='Qari OCR')


def _authorise(authorization: str | None) -> None:
    import hmac

    expected = f'Bearer {API_SECRET}'
    # compare_digest, not ==, so the comparison does not leak the secret's
    # length or prefix through timing.
    if not authorization or not hmac.compare_digest(authorization, expected):
        raise HTTPException(status_code=401, detail='bad or missing bearer token')


@app.get('/health')
def health():
    """Unauthenticated on purpose: it reveals nothing and lets a caller check
    whether the tunnel is alive before sending a document."""
    return {
        'status': 'ok',
        'model': MODEL_ID,
        'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        'vram_allocated_gb': round(torch.cuda.memory_allocated() / 1e9, 2),
    }


@app.post('/ocr')
async def ocr(
    file: UploadFile = File(...),
    first_page: int = Form(0),
    last_page: int = Form(-1),
    authorization: str | None = Header(default=None),
):
    _authorise(authorization)

    data = await file.read()

    if not data:
        raise HTTPException(status_code=400, detail='empty upload')

    if len(data) > MAX_UPLOAD_MB * 1024 * 1024:
        raise HTTPException(
            status_code=413,
            detail=f'{len(data) / 1e6:.1f} MB exceeds the {MAX_UPLOAD_MB} MB limit',
        )

    last = None if last_page < 0 else last_page

    if last is not None and last - first_page + 1 > MAX_PAGES_PER_REQUEST:
        raise HTTPException(
            status_code=400,
            detail=f'at most {MAX_PAGES_PER_REQUEST} pages per request; '
                   'send the document in batches',
        )

    if last is None:
        last = first_page + MAX_PAGES_PER_REQUEST - 1

    try:
        return {'filename': file.filename, **read_pdf(data, first_page, last)}
    except Exception as exc:
        raise HTTPException(status_code=500, detail=f'{type(exc).__name__}: {exc}')

print('api defined')


## 7. Start the server and open the tunnel

Run this once. Re-running it without restarting the runtime will fail on the port
already being in use — use *Runtime → Restart* if you need to change anything above.


In [ ]:
import threading, time
import uvicorn
from pyngrok import conf, ngrok

conf.get_default().auth_token = NGROK_AUTHTOKEN

# Close anything this session left open, or restarts accumulate dead tunnels
# until the free-tier limit rejects the next one.
for tunnel in ngrok.get_tunnels():
    ngrok.disconnect(tunnel.public_url)

server = uvicorn.Server(uvicorn.Config(app, host='0.0.0.0', port=PORT, log_level='warning'))
threading.Thread(target=server.run, daemon=True).start()
time.sleep(3)

PUBLIC_URL = ngrok.connect(PORT, 'http').public_url

print('=' * 70)
print(f'  {PUBLIC_URL}')
print('=' * 70)
print('Give that URL and your API_SECRET to the client. Both change on restart.')


## 8. Check it works, from inside the notebook


In [ ]:
import requests

print(requests.get(f'{PUBLIC_URL}/health', timeout=30).json())

# Refused without the token -- confirms the door is actually shut.
print('no token ->', requests.post(f'{PUBLIC_URL}/ocr', files={'file': ('x.pdf', b'%PDF-')},
                                   timeout=30).status_code, '(expect 401)')


## 9. Send a real PDF

Two ways to get a file in:

- **Small file** — drag it into the file browser (folder icon, left-hand side).
  It lands in `/content/`. It is deleted when the session ends.
- **Large file, or one you use repeatedly** — mount Drive and read it from there:

  ```python
  from google.colab import drive
  drive.mount('/content/drive')
  PDF_PATH = '/content/drive/MyDrive/your.pdf'
  ```

Note the request below asks for pages 0–1 only. A 274-page book in one request
will time out long before it finishes.


In [ ]:
PDF_PATH = '/content/sample.pdf'  # <- change me

import os

if not os.path.exists(PDF_PATH):
    print(f'{PDF_PATH} not found -- upload a PDF using the file browser on the left.')
else:
    with open(PDF_PATH, 'rb') as handle:
        response = requests.post(
            f'{PUBLIC_URL}/ocr',
            headers={'Authorization': f'Bearer {API_SECRET}'},
            files={'file': (os.path.basename(PDF_PATH), handle, 'application/pdf')},
            data={'first_page': 0, 'last_page': 1},
            timeout=600,
        )

    response.raise_for_status()
    result = response.json()

    print(f"{result['page_count']} pages in the file, "
          f"{len(result['pages'])} read in {result['seconds']}s\n")

    for page in result['pages']:
        print(f"--- page {page['page'] + 1}  ({page['seconds']}s) ---")
        print(page['text'][:600])
        print()


## 10. Keep it alive

Colab stops a notebook whose tab has been idle for about 90 minutes, and every
session ends at ~12 hours regardless. The cell below prints a heartbeat so the
session counts as active; **leave the tab open**.

Stop it with the interrupt button when you are done.


In [ ]:
import datetime, time

print(f'serving {PUBLIC_URL} -- interrupt this cell to stop keeping the session warm')

try:
    while True:
        time.sleep(600)
        stamp = datetime.datetime.now().strftime('%H:%M:%S')
        alive = requests.get(f'{PUBLIC_URL}/health', timeout=30).ok
        print(f"[{stamp}] tunnel {'up' if alive else 'DOWN'}")
except KeyboardInterrupt:
    print('stopped keeping alive; the server is still running')
